<a href="https://colab.research.google.com/github/DSon24/Interpretability-study-of-Artificial-Hippocampus-Networks/blob/main/fitted-jlens" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q -U git+https://github.com/anthropics/jacobian-lens.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import torch
import transformers
import jlens

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("J-Lens: OK")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch: 2.11.0+cu128
Transformers: 5.15.0
J-Lens: OK
GPU: Tesla T4


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()

print("Model loaded ✅")
print("Hidden size:", model.config.hidden_size)
print("Layers:", model.config.num_hidden_layers)
print("Device:", model.device)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded ✅
Hidden size: 2048
Layers: 36
Device: cuda:0


In [5]:
import jlens

jl_model = jlens.from_hf(model, tokenizer)

print("J-Lens wrapper loaded ✅")
print("Layers:", jl_model.n_layers)
print("d_model:", jl_model.d_model)

J-Lens wrapper loaded ✅
Layers: 36
d_model: 2048


In [6]:
prompts = [
    "The capital of France is Paris. Machine learning models process information through multiple layers."
]

lens = jlens.fit(
    jl_model,
    prompts=prompts,
    source_layers=[18],
    dim_batch=8,
    max_seq_len=64,
    skip_first=4,
)

print("J-Lens fit complete ✅")
print("J18 shape:", lens.jacobians[18].shape)
print("J18 norm:", lens.jacobians[18].float().norm().item())

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


J-Lens fit complete ✅
J18 shape: torch.Size([2048, 2048])
J18 norm: 66.8904800415039


In [7]:
torch.save(
    lens.jacobians[18].cpu(),
    "/content/J18_qwen25_3b.pt"
)

print("Saved J18 ✅")

Saved J18 ✅


In [8]:
from google.colab import files
files.download("/content/J18_qwen25_3b.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>